In [1]:
# %pip install azure-ai-documentintelligence --pre

In [3]:
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentAnalysisFeature, AnalyzeResult, AnalyzeDocumentRequest

In [8]:
# For how to obtain the endpoint and key, please see PREREQUISITES above.
endpoint = "https://docintell-bluesky.cognitiveservices.azure.com/"
key = "658ac4bab51444f9ac8254efa8678718"

document_intelligence_client = DocumentIntelligenceClient(endpoint=endpoint, credential=AzureKeyCredential(key))


formUrl = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-layout.pdf"


poller = document_intelligence_client.begin_analyze_document(
        "prebuilt-read",
        AnalyzeDocumentRequest(url_source=formUrl),
        features=[DocumentAnalysisFeature.LANGUAGES]
    )       

result: AnalyzeResult = poller.result()

In [10]:
def get_words(page, line):
    result = []
    for word in page.words:
        if _in_span(word, line.spans):
            result.append(word)
    return result

# To learn the detailed concept of "span" in the following codes, visit: https://aka.ms/spans 
def _in_span(word, spans):
    for span in spans:
        if word.span.offset >= span.offset and (word.span.offset + word.span.length) <= (span.offset + span.length):
            return True
    return False


In [11]:
print("----Languages detected in the document----")
if result.languages is not None:
    for language in result.languages:
        print(f"Language code: '{language.locale}' with confidence {language.confidence}")

# To learn the detailed concept of "bounding polygon" in the following content, visit: https://aka.ms/bounding-region
# Analyze pages.
for page in result.pages:
    print(f"----Analyzing document from page #{page.page_number}----")
    print(f"Page has width: {page.width} and height: {page.height}, measured with unit: {page.unit}")

    # Analyze lines.
    if page.lines:
        for line_idx, line in enumerate(page.lines):
            words = get_words(page, line)
            print(
                f"...Line # {line_idx} has {len(words)} words and text '{line.content}' within bounding polygon '{line.polygon}'"
            )

            # Analyze words.
            for word in words:
                print(f"......Word '{word.content}' has a confidence of {word.confidence}")
    
# Analyze paragraphs.
if result.paragraphs:
    print(f"----Detected #{len(result.paragraphs)} paragraphs in the document----")
    for paragraph in result.paragraphs:
        print(f"Found paragraph within {paragraph.bounding_regions} bounding region")
        print(f"...with content: '{paragraph.content}'")

print("----------------------------------------")

----Languages detected in the document----
Language code: 'en' with confidence 1
Language code: 'en' with confidence 0.2
Language code: 'ja' with confidence 0.8
Language code: 'ja' with confidence 0.6
Language code: 'en' with confidence 0.95
Language code: 'en' with confidence 0.6
Language code: 'mg' with confidence 0.2
Language code: 'en' with confidence 0.7
Language code: 'en' with confidence 0.99
Language code: 'en' with confidence 0.9
Language code: 'en' with confidence 0.5
----Analyzing document from page #1----
Page has width: 8.5 and height: 11, measured with unit: inch
...Line # 0 has 2 words and text 'UNITED STATES' within bounding polygon '[3.4669, 0.6636, 5.0236, 0.6636, 5.0236, 0.8403, 3.4669, 0.8403]'
......Word 'UNITED' has a confidence of 0.995
......Word 'STATES' has a confidence of 0.995
...Line # 1 has 4 words and text 'SECURITIES AND EXCHANGE COMMISSION' within bounding polygon '[2.1728, 0.8785, 6.3177, 0.8785, 6.3177, 1.0647, 2.1728, 1.0647]'
......Word 'SECURITIES'